In [1]:
import osmnx as ox
import pandas as pd
import geopandas as gpd


##using osmnx to get example table with cinemas

place_name = "Berlin, Germany"
tags = {"amenity": "cinema"}

# This works in OSMnx 2.x
gdf = ox.features_from_place(place_name, tags)



In [2]:
#example of selecting needed columns
columns = ['name', 'amenity', 'geometry', 'addr:street', 'addr:housenumber', 'addr:postcode', 'website', 'operator']
cinema = gdf[columns]
cinema.head()

name amenity                   geometry  \
element id                                                               
node    29040741   Filmrauschpalast  cinema  POINT (13.35962 52.53438)   
        79615774           CineStar  cinema    POINT (13.286 52.58392)   
        101186634   UCI Am Eastgate  cinema  POINT (13.54203 52.54175)   
        251559871      Kino Intimes  cinema   POINT (13.4582 52.51291)   
        256754256           Babylon  cinema   POINT (13.41144 52.5259)   

                             addr:street addr:housenumber addr:postcode  \
element id                                                                
node    29040741          Lehrter Straße               35         10557   
        79615774                     NaN              NaN           NaN   
        101186634        Märkische Allee          176-178         12681   
        251559871      Boxhagener Straße              107         10245   
        256754256  Rosa-Luxemburg-Straße               30         10178   

                                                             website  \
element id                                                             
node    29040741                                                 NaN   
        79615774                                                 NaN   
        101186634  https://www.uci-kinowelt.de/kinoprogramm/berli...   
        251559871                                                NaN   
        256754256                                                NaN   

                                 operator  
element id                                 
node    29040741   Filmrausch Moabit e.V.  
        79615774                      NaN  
        101186634                     NaN  
        251559871                     NaN  
        256754256                     NaN

In [3]:
# Load official Berlin districts GeoDataFrame from lor_ortsteile.geojson
berlin_districts_gdf = gpd.read_file("lor_ortsteile.geojson")

In [4]:
# Spatial join: matching your cinemas with district and neighborhoods
cinema_with_districts = gpd.sjoin(
    cinema,
    berlin_districts_gdf[["BEZIRK", "spatial_name", "OTEIL","geometry"]],
    how="left",
    predicate="within"
)
cinema_with_districts.head()

name amenity                   geometry  \
element id                                                               
node    29040741   Filmrauschpalast  cinema  POINT (13.35962 52.53438)   
        79615774           CineStar  cinema    POINT (13.286 52.58392)   
        101186634   UCI Am Eastgate  cinema  POINT (13.54203 52.54175)   
        251559871      Kino Intimes  cinema   POINT (13.4582 52.51291)   
        256754256           Babylon  cinema   POINT (13.41144 52.5259)   

                             addr:street addr:housenumber addr:postcode  \
element id                                                                
node    29040741          Lehrter Straße               35         10557   
        79615774                     NaN              NaN           NaN   
        101186634        Märkische Allee          176-178         12681   
        251559871      Boxhagener Straße              107         10245   
        256754256  Rosa-Luxemburg-Straße               30         10178   

                                                             website  \
element id                                                             
node    29040741                                                 NaN   
        79615774                                                 NaN   
        101186634  https://www.uci-kinowelt.de/kinoprogramm/berli...   
        251559871                                                NaN   
        256754256                                                NaN   

                                 operator  index_right  \
element id                                               
node    29040741   Filmrausch Moabit e.V.            1   
        79615774                      NaN           86   
        101186634                     NaN           70   
        251559871                     NaN            6   
        256754256                     NaN            0   

                                     BEZIRK spatial_name           OTEIL  
element id                                                                
node    29040741                      Mitte         0102          Moabit  
        79615774              Reinickendorf         1202           Tegel  
        101186634       Marzahn-Hellersdorf         1001         Marzahn  
        251559871  Friedrichshain-Kreuzberg         0201  Friedrichshain  
        256754256                     Mitte         0101           Mitte

In [5]:
##just renaming columns for proper schema
cinema_with_districts = cinema_with_districts.rename(columns={
    "BEZIRK": "district",
    "OTEIL": "neighborhood",
    "spatial_name": "neighborhood_id"
}).drop(columns=["index_right"])  # drop district_number if not needed
cinema_with_districts.head()

name amenity                   geometry  \
element id                                                               
node    29040741   Filmrauschpalast  cinema  POINT (13.35962 52.53438)   
        79615774           CineStar  cinema    POINT (13.286 52.58392)   
        101186634   UCI Am Eastgate  cinema  POINT (13.54203 52.54175)   
        251559871      Kino Intimes  cinema   POINT (13.4582 52.51291)   
        256754256           Babylon  cinema   POINT (13.41144 52.5259)   

                             addr:street addr:housenumber addr:postcode  \
element id                                                                
node    29040741          Lehrter Straße               35         10557   
        79615774                     NaN              NaN           NaN   
        101186634        Märkische Allee          176-178         12681   
        251559871      Boxhagener Straße              107         10245   
        256754256  Rosa-Luxemburg-Straße               30         10178   

                                                             website  \
element id                                                             
node    29040741                                                 NaN   
        79615774                                                 NaN   
        101186634  https://www.uci-kinowelt.de/kinoprogramm/berli...   
        251559871                                                NaN   
        256754256                                                NaN   

                                 operator                  district  \
element id                                                            
node    29040741   Filmrausch Moabit e.V.                     Mitte   
        79615774                      NaN             Reinickendorf   
        101186634                     NaN       Marzahn-Hellersdorf   
        251559871                     NaN  Friedrichshain-Kreuzberg   
        256754256                     NaN                     Mitte   

                  neighborhood_id    neighborhood  
element id                                         
node    29040741             0102          Moabit  
        79615774             1202           Tegel  
        101186634            1001         Marzahn  
        251559871            0201  Friedrichshain  
        256754256            0101           Mitte

In [6]:
# District mapping (official codes as strings)
district_mapping = {
    'Mitte': '11001001',
    'Friedrichshain-Kreuzberg': '11002002',
    'Pankow': '11003003',
    'Charlottenburg-Wilmersdorf': '11004004',
    'Spandau': '11005005',
    'Steglitz-Zehlendorf': '11006006',
    'Tempelhof-Schöneberg': '11007007',
    'Neukölln': '11008008',
    'Treptow-Köpenick': '11009009',
    'Marzahn-Hellersdorf': '11010010',
    'Lichtenberg': '11011011',
    'Reinickendorf': '11012012'
}

# Apply mapping to create district_id column (string)
cinema_with_districts['district_id'] = cinema_with_districts['district'].map(district_mapping).astype(str)

# (Optional) Check if some districts were not mapped
#unmapped = df[~df['district'].isin(district_mapping.keys())]['district'].unique()
#if len(unmapped) > 0:
    #print("⚠️ Unmapped districts found:", unmapped)
cinema_with_districts.head()

name amenity                   geometry  \
element id                                                               
node    29040741   Filmrauschpalast  cinema  POINT (13.35962 52.53438)   
        79615774           CineStar  cinema    POINT (13.286 52.58392)   
        101186634   UCI Am Eastgate  cinema  POINT (13.54203 52.54175)   
        251559871      Kino Intimes  cinema   POINT (13.4582 52.51291)   
        256754256           Babylon  cinema   POINT (13.41144 52.5259)   

                             addr:street addr:housenumber addr:postcode  \
element id                                                                
node    29040741          Lehrter Straße               35         10557   
        79615774                     NaN              NaN           NaN   
        101186634        Märkische Allee          176-178         12681   
        251559871      Boxhagener Straße              107         10245   
        256754256  Rosa-Luxemburg-Straße               30         10178   

                                                             website  \
element id                                                             
node    29040741                                                 NaN   
        79615774                                                 NaN   
        101186634  https://www.uci-kinowelt.de/kinoprogramm/berli...   
        251559871                                                NaN   
        256754256                                                NaN   

                                 operator                  district  \
element id                                                            
node    29040741   Filmrausch Moabit e.V.                     Mitte   
        79615774                      NaN             Reinickendorf   
        101186634                     NaN       Marzahn-Hellersdorf   
        251559871                     NaN  Friedrichshain-Kreuzberg   
        256754256                     NaN                     Mitte   

                  neighborhood_id    neighborhood district_id  
element id                                                     
node    29040741             0102          Moabit    11001001  
        79615774             1202           Tegel    11012012  
        101186634            1001         Marzahn    11010010  
        251559871            0201  Friedrichshain    11002002  
        256754256            0101           Mitte    11001001